LAB 12

In [28]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("lab12").master("local[*]").getOrCreate()

df = spark.read.parquet("bigdata/silver/transactions_enriched.parquet")

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

indexer = StringIndexer(inputCol="segment", outputCol="segment_idx")
assembler = VectorAssembler(inputCols=["amount", "risk_score", "credit_score", "segment_idx"], outputCol="features")

df_ml = df.withColumn("label", df.is_fraud.cast("integer"))
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
print("treino:", train.count(), "· teste:", test.count())

lr = LogisticRegression(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[indexer, assembler, lr])
modelo = pipeline.fit(train)

predicoes = modelo.transform(test)
avaliador = BinaryClassificationEvaluator(labelCol="label")
auc = avaliador.evaluate(predicoes)
print(f"AUC no teste: {auc:.3f}")

predicoes.select("transaction_id", "amount", "segment", "label", "prediction", "probability").show(10, truncate=False)

lr_model = modelo.stages[-1]
coefs = lr_model.coefficients
nomes = ["amount", "risk_score", "credit_score", "segment_idx"]
for nome, coef in zip(nomes, coefs):
    print(f"{nome}: {coef:.4f}")

treino: 79901 · teste: 20099
AUC no teste: 0.768
+--------------+---------+---------+-----+----------+------------------------------------------+
|transaction_id|amount   |segment  |label|prediction|probability                               |
+--------------+---------+---------+-----+----------+------------------------------------------+
|3             |93.935265|High-Risk|0    |0.0       |[0.9495784233506188,0.05042157664938118]  |
|7             |140.57378|Premium  |0    |0.0       |[0.9944785942804952,0.005521405719504768] |
|9             |10.0     |High-Risk|0    |0.0       |[0.9338760980125079,0.06612390198749207]  |
|14            |81.39329 |Standard |0    |0.0       |[0.975478357338241,0.024521642661759047]  |
|20            |34.814598|Premium  |0    |0.0       |[0.9950050063834911,0.004994993616508903] |
|24            |26.91095 |Premium  |0    |0.0       |[0.988864583273263,0.011135416726737013]  |
|30            |130.24501|Standard |0    |0.0       |[0.9867809580635087,0.013